In [2]:
import sys
print(sys.version)
print(sys.executable)

3.14.2 (main, Sep 10 2026, 12:33:04) [GCC 13.3.0]
/workspaces/ai-engineering-buildcamp/.venv/bin/python


In [3]:
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import AppendableIndex

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)

files = reader.read()

parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

index = AppendableIndex(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)

index.fit(chunked_docs)

In [4]:
def search(query):
    results = index.search(
        query=query,
        num_results=5
    )
    return results

In [5]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the documentation database for relevant results based on a query string.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query to look up in the index"
            }
        },
        "required": ["query"]
    }
}

In [6]:
def add_entry(filename, title, description, content):
    entry = {
        "start": 0,
        "content": content,
        "title": title,
        "description": description,
        "filename": filename,
    }
    index.append(entry)
    return "OK"

In [7]:
add_entry_tool = {
    "type": "function",
    "name": "add_entry",
    "description": "Add a new documentation entry to the index.",
    "parameters": {
        "type": "object",
        "properties": {
            "filename": {
                "type": "string",
                "description": "The source filename associated with the entry"
            },
            "title": {
                "type": "string",
                "description": "The title of the documentation entry"
            },
            "description": {
                "type": "string",
                "description": "A short description summarizing the entry"
            },
            "content": {
                "type": "string",
                "description": "The full content of the documentation entry"
            }
        },
        "required": ["filename", "title", "description", "content"]
    }
}


In [8]:
instructions = """
You're a documentation assistant.

Answer the user question using the documentation knowledge base.

Make 3 iterations:

1) in the first iteration, perform one search

2) in the second interation, analyze the results from the previous search
   and perform 2 more searches

3) synthesise the results into the output

IMPORTANT: at each step, give an explanation of why you want to perform
search for this particular search query. It should be 2-3 sentences explaining
the logic of your decision.

Use only facts from the knowledge base when answering.
If you cannot find the answer, inform the user.

Our knowledge base is entirely about Evidently, so you don't need to
include the word 'evidently' in search results
"""

In [10]:
from toyaikit.tools import Tools

agent_tools = Tools()

agent_tools.add_tool(search, search_tool)
agent_tools.add_tool(add_entry, add_entry_tool)

agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the documentation database for relevant results based on a query string.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'The search query to look up in the index'}},
   'required': ['query']}},
 {'type': 'function',
  'name': 'add_entry',
  'description': 'Add a new documentation entry to the index.',
  'parameters': {'type': 'object',
   'properties': {'filename': {'type': 'string',
     'description': 'The source filename associated with the entry'},
    'title': {'type': 'string',
     'description': 'The title of the documentation entry'},
    'description': {'type': 'string',
     'description': 'A short description summarizing the entry'},
    'content': {'type': 'string',
     'description': 'The full content of the documentation entry'}},
   'required': ['filename', 'title', 'description', 'content']}}]

In [11]:
from toyaikit.chat.runners import OpenAIResponsesRunner
from toyaikit.llm import OpenAIClient
from openai import OpenAI

llm_client = OpenAIClient(
    model="gpt-4o-mini",
    client=OpenAI()
)

agent = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    llm_client=llm_client
)

In [12]:
result = agent.loop("how can I create evidently dahsbord?")

In [13]:
print(result.last_message)

Based on the gathered information, here’s how to create a dashboard effectively:

### 1. Creating a Dashboard
To create a dashboard, you can utilize either the **User Interface (UI)** or the **Python API**.

### **User Interface (UI)**
- **Enter Edit Mode**: Click on the "Edit" button on the dashboard interface.
- **Add Tabs**: You can create multiple tabs to organize panels.
  - Click the plus icon to add a new tab and choose a name.
- **Add Panels**: Follow these steps:
  1. Click on the "Add Panel" button.
  2. Choose the type of panel (e.g., Text, Counter, Pie Chart, Line Plot).
  3. Configure the panel settings using the prompts.
  4. Click "Save" when you’re done, selecting the appropriate tab for it.
  5. Use metrics from your reports to populate the panels.

### **Python API**
To create a dashboard programmatically, you can use the following steps:
1. **Set Up**: First, connect to your project and ensure you have reports with evaluation results.
2. **Add a Tab**:
   ```python
 

In [14]:
from toyaikit.chat.interface import IPythonChatInterface
from toyaikit.chat.runners import DisplayingRunnerCallback

chat_interface = IPythonChatInterface()
runner_callback = DisplayingRunnerCallback(
    chat_interface=chat_interface
)

In [15]:
result = agent.loop(
    "how can I create evidently dahsbard?",
    callback=runner_callback
)

-> Response received


-> Response received


-> Response received


In [16]:
result2 = agent.loop(
    "show me the code",
    callback=runner_callback,
    previous_messages=result.all_messages
)

-> Response received


In [17]:
agent = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=llm_client
)

In [18]:
result = agent.run()

-> Response received


-> Response received


-> Response received


KeyboardInterrupt: Interrupted by user

In [19]:
result.cost.total_cost

Decimal('0.00290565')